# Lấy danh sách môn bắt buộc / tự chọn theo ngành + học kỳ

Đọc các file `data/processed/curriculum_graph/{NGANH}_curriculum.json` (5 ngành: CS, IS, DS, SE, IT) và tách môn của 1 học kỳ theo loại (`bat_buoc` / `tu_chon`).

**Schema mỗi file** (rút gọn):
```json
{
  "nganh": "CS",
  "hoc_ky": [
    {
      "hk_so": 1,
      "tong_tc": 11,
      "hoc_phan": [
        {"ma_mon": "002793", "ten_mon": "...", "loai": "bat_buoc", "so_tc": 2, ...}
      ]
    }
  ]
}
```

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Literal

import pandas as pd

# Notebook ở notebooks/, dữ liệu ở data/processed/curriculum_graph/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CURRICULUM_DIR = ROOT / "data" / "processed" / "curriculum_graph"

NGANH_HOP_LE = {"CS", "IS", "DS", "SE", "IT"}
LoaiMon = Literal["bat_buoc", "tu_chon", "all"]

print(f"CURRICULUM_DIR = {CURRICULUM_DIR}")
print(f"Tồn tại: {CURRICULUM_DIR.exists()}")

CURRICULUM_DIR = e:\CK_NLP\chatbot-tu-van-chien-luoc-hoc-tap-khoa-cntt\data\processed\curriculum_graph
Tồn tại: True


In [2]:
def load_curriculum(nganh: str) -> dict:
    """Đọc file `{nganh}_curriculum.json` và trả về dict gốc.

    Args:
        nganh: Mã ngành (CS / IS / DS / SE / IT), không phân biệt hoa thường.

    Returns:
        Dict chứa keys `nganh` và `hoc_ky`.

    Raises:
        ValueError: nếu ngành không hợp lệ.
        FileNotFoundError: nếu file curriculum không tồn tại.
    """
    nganh = nganh.upper()
    if nganh not in NGANH_HOP_LE:
        raise ValueError(f"Ngành '{nganh}' không hợp lệ. Chọn 1 trong: {sorted(NGANH_HOP_LE)}")

    path = CURRICULUM_DIR / f"{nganh}_curriculum.json"
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path}")

    with path.open(encoding="utf-8") as f:
        return json.load(f)

In [3]:
def lay_mon_theo_hk(
    nganh: str,
    hk_so: int,
    loai: LoaiMon = "all",
    bo_mon_khong_gpa: bool = False,
) -> pd.DataFrame:
    """Trả về DataFrame các môn của 1 ngành ở 1 học kỳ, lọc theo loại.

    Args:
        nganh: Mã ngành (CS / IS / DS / SE / IT).
        hk_so: Số học kỳ (1–9).
        loai: 'bat_buoc' | 'tu_chon' | 'all'. Mặc định 'all' lấy hết.
        bo_mon_khong_gpa: Nếu True, loại các môn `khong_tinh_gpa=True`
            (GDTC, Quốc phòng, Chứng chỉ Tiếng Anh, ...).

    Returns:
        DataFrame các cột: ma_mon, ten_mon, loai, so_tc, so_tiet_lt, so_tiet_th,
        khong_tinh_gpa, dieu_kien, loai_dieu_kien.

    Raises:
        ValueError: nếu `hk_so` không có trong curriculum hoặc `loai` không hợp lệ.
    """
    if loai not in {"bat_buoc", "tu_chon", "all"}:
        raise ValueError(f"loai='{loai}' không hợp lệ. Chọn: bat_buoc | tu_chon | all")

    data = load_curriculum(nganh)

    hk_dict = next((hk for hk in data["hoc_ky"] if hk["hk_so"] == hk_so), None)
    if hk_dict is None:
        ds_hk = sorted({hk["hk_so"] for hk in data["hoc_ky"]})
        raise ValueError(f"Ngành {nganh.upper()} không có HK{hk_so}. Các HK có: {ds_hk}")

    rows = hk_dict["hoc_phan"]
    if loai != "all":
        rows = [m for m in rows if m["loai"] == loai]
    if bo_mon_khong_gpa:
        rows = [m for m in rows if not m.get("khong_tinh_gpa", False)]

    df = pd.DataFrame(rows)
    # Sắp thứ tự cột cho dễ đọc
    if not df.empty:
        cols_uu_tien = [
            "ma_mon", "ten_mon", "loai", "so_tc",
            "so_tiet_lt", "so_tiet_th", "khong_tinh_gpa",
            "dieu_kien", "loai_dieu_kien",
        ]
        cols = [c for c in cols_uu_tien if c in df.columns] + [
            c for c in df.columns if c not in cols_uu_tien
        ]
        df = df[cols]
    return df

## Ví dụ sử dụng

In [4]:
# Ví dụ 1: Tất cả môn bắt buộc của CS HK3
df_bb = lay_mon_theo_hk("CS", hk_so=3, loai="bat_buoc")
print(f"Số môn bắt buộc CS HK3: {len(df_bb)} | tổng TC: {df_bb['so_tc'].sum()}")
df_bb

Số môn bắt buộc CS HK3: 5 | tổng TC: 17


,ma_mon,ten_mon,loai,so_tc,so_tiet_lt,so_tiet_th,khong_tinh_gpa,dieu_kien,loai_dieu_kien,ma_hoc_phan
0,001508,Cấu trúc rời rạc,bat_buoc,3,45,0,False,[],None,4220001508
1,001611,Cấu trúc dữ liệu và giải thuật,bat_buoc,4,45,30,False,[004247],a,4220001611
2,001922,Hệ cơ sở dữ liệu,bat_buoc,4,45,30,False,[002793],a,4220001922
3,003595,Toán cao cấp 2,bat_buoc,2,30,0,False,[],None,4220003595
4,015256,Anh văn 2,bat_buoc,4,60,0,False,[],None,4220015256


In [5]:
# Ví dụ 2: Tất cả môn tự chọn của CS HK3
df_tc = lay_mon_theo_hk("CS", hk_so=3, loai="tu_chon")
print(f"Số môn tự chọn CS HK3: {len(df_tc)} | tổng TC: {df_tc['so_tc'].sum()}")
df_tc

Số môn tự chọn CS HK3: 9 | tổng TC: 24


,ma_mon,ten_mon,loai,so_tc,so_tiet_lt,so_tiet_th,khong_tinh_gpa,dieu_kien,loai_dieu_kien,ma_hoc_phan
0,001479,Địa lí kinh tế,tu_chon,3,45,0,False,[],None,4220001479
1,003582,Kỹ năng xây dựng kế hoạch,tu_chon,3,45,0,False,[],None,4220003582
2,003877,Môi trường và con người,tu_chon,3,45,0,False,[],None,4220003877
3,014233,Ngôn ngữ Python,tu_chon,2,0,60,False,[],None,4220014233
4,014234,Tính toán số & Matlab,tu_chon,2,0,60,False,[001782],a,4220014234
5,014235,Ngôn ngữ R,tu_chon,2,0,60,False,[],None,4220014235
6,015395,Ứng dụng hóa học trong Công nghiệp,tu_chon,3,45,0,False,[],None,4220015395
7,015396,Ứng dụng 5S và Kaizen trong sản xuất,tu_chon,3,45,0,False,[],None,4220015396
8,015400,Công nghệ thông tin trong chuyển đổi số,tu_chon,3,45,0,False,[],None,4220015400


In [6]:
# Ví dụ 3: Tất cả môn (bắt buộc + tự chọn) của IS HK5, bỏ môn không tính GPA
df_all = lay_mon_theo_hk("IS", hk_so=5, loai="all", bo_mon_khong_gpa=True)
print(f"Số môn IS HK5 (đã bỏ không-GPA): {len(df_all)} | tổng TC: {df_all['so_tc'].sum()}")
df_all

Số môn IS HK5 (đã bỏ không-GPA): 9 | tổng TC: 27


,ma_mon,ten_mon,loai,so_tc,so_tiet_lt,so_tiet_th,khong_tinh_gpa,dieu_kien,loai_dieu_kien,ma_hoc_phan
0,001815,Nhập môn an toàn thông tin,bat_buoc,3,45,0,False,[],None,4220001815
1,002399,Hệ Thống và Công nghệ Web,bat_buoc,3,30,30,False,[],None,4220002399
2,003172,Pháp luật đại cương,bat_buoc,2,30,0,False,[],None,4220003172
3,003791,Phân tích thiết kế hệ thống,bat_buoc,3,30,30,False,[],None,4220003791
4,004022,Lập trình phân tích dữ liệu 1,bat_buoc,3,30,30,False,[004247],b,4220004022
5,015257,Anh văn 3,bat_buoc,4,60,0,False,[],None,4220015257
6,015061,Giao dịch định lượng,tu_chon,3,30,30,False,[],None,4220015061
7,015062,Phân tích định lượng,tu_chon,3,45,0,False,[004350],b,4220015062
8,015065,Trực quan hóa dữ liệu,tu_chon,3,30,30,False,[],None,4220015065


In [7]:
# Ví dụ 4: Bảng tổng hợp số môn bắt buộc / tự chọn theo HK của 1 ngành
def thong_ke_theo_hk(nganh: str) -> pd.DataFrame:
    """Đếm số môn bắt buộc / tự chọn và tổng TC mỗi HK của 1 ngành."""
    data = load_curriculum(nganh)
    rows = []
    for hk in data["hoc_ky"]:
        hp = hk["hoc_phan"]
        bb = [m for m in hp if m["loai"] == "bat_buoc"]
        tc = [m for m in hp if m["loai"] == "tu_chon"]
        rows.append({
            "hk_so": hk["hk_so"],
            "so_mon_bat_buoc": len(bb),
            "tc_bat_buoc": sum(m["so_tc"] for m in bb),
            "so_mon_tu_chon": len(tc),
            "tc_tu_chon": sum(m["so_tc"] for m in tc),
            "tong_tc": hk.get("tong_tc"),
        })
    return pd.DataFrame(rows)

thong_ke_theo_hk("CS")

,hk_so,so_mon_bat_buoc,tc_bat_buoc,so_mon_tu_chon,tc_tu_chon,tong_tc
0,1,8,17,0,0,11
1,2,6,18,5,15,15
2,3,5,17,9,24,22
3,4,7,20,3,9,23
4,5,5,16,6,21,23
5,6,6,17,4,12,20
6,7,4,11,10,30,17
7,8,4,11,7,21,17
8,9,2,13,0,0,13


In [8]:
# Ví dụ 5: So sánh môn tự chọn HK6 giữa 5 ngành
for ng in ["CS", "IS", "DS", "SE", "IT"]:
    try:
        df = lay_mon_theo_hk(ng, hk_so=6, loai="tu_chon")
        print(f"{ng} HK6 — {len(df)} môn tự chọn, {df['so_tc'].sum()} TC")
    except ValueError as e:
        print(f"{ng}: {e}")

CS HK6 — 4 môn tự chọn, 12 TC
IS HK6 — 6 môn tự chọn, 18 TC
DS HK6 — 7 môn tự chọn, 21 TC
SE HK6 — 3 môn tự chọn, 9 TC
IT HK6 — 3 môn tự chọn, 9 TC


## Lấy danh sách mã môn tích lũy từ HK1 → HK tùy chọn

Trả về **list mã môn 6 chữ số** (kiểu `['004247', '001782', ...]`) cộng dồn từ HK1 đến `hk_max`, tách riêng **bắt buộc** và **tự chọn**.

In [9]:
def lay_ma_mon_tich_luy(
    nganh: str,
    hk_max: int,
    bo_mon_khong_gpa: bool = False,
) -> dict[str, list[str]]:
    """Lấy list mã môn tích lũy từ HK1 đến `hk_max`, tách BB / TC.

    Args:
        nganh: Mã ngành (CS / IS / DS / SE / IT).
        hk_max: Học kỳ cuối cần lấy (bao gồm). Mọi HK <= hk_max sẽ được gộp.
        bo_mon_khong_gpa: Nếu True, loại các môn `khong_tinh_gpa=True`
            (GDTC, Quốc phòng, Chứng chỉ Tiếng Anh, ...).

    Returns:
        Dict 2 keys:
            - 'bat_buoc': list[str] mã môn bắt buộc (6 chữ số).
            - 'tu_chon':  list[str] mã môn tự chọn (6 chữ số).
        Mã môn giữ thứ tự xuất hiện theo HK tăng dần, không trùng lặp.
    """
    data = load_curriculum(nganh)
    bb_list: list[str] = []
    tc_list: list[str] = []
    seen_bb: set[str] = set()
    seen_tc: set[str] = set()

    for hk in sorted(data["hoc_ky"], key=lambda x: x["hk_so"]):
        if hk["hk_so"] > hk_max:
            break
        for m in hk["hoc_phan"]:
            if bo_mon_khong_gpa and m.get("khong_tinh_gpa", False):
                continue
            ma = str(m["ma_mon"]).zfill(6)
            if m["loai"] == "bat_buoc" and ma not in seen_bb:
                bb_list.append(ma)
                seen_bb.add(ma)
            elif m["loai"] == "tu_chon" and ma not in seen_tc:
                tc_list.append(ma)
                seen_tc.add(ma)

    return {"bat_buoc": bb_list, "tu_chon": tc_list}

In [11]:
# Ví dụ: list mã môn tích lũy HK1 -> HK5 của ngành CS
NGANH = "CS"
HK_MAX = 2

result = lay_ma_mon_tich_luy(NGANH, hk_max=HK_MAX, bo_mon_khong_gpa=False)

print(f"=== {NGANH} HK1 -> HK{HK_MAX} ===")
print(f"\nBắt buộc ({len(result['bat_buoc'])} môn):")
print(result["bat_buoc"])

print(f"\nTự chọn ({len(result['tu_chon'])} môn):")
print(result["tu_chon"])

=== CS HK1 -> HK2 ===

Bắt buộc (14 môn):
['002793', '003573', '003575', '003696', '003801', '004247', '013801', '015217', '001782', '003604', '003630', '003949', '013802', '015255']

Tự chọn (5 môn):
['003605', '003631', '003697', '003783', '003822']


In [ ]:
# Ví dụ: in list cho cả 5 ngành tới HK4
HK_MAX = 4
for ng in ["CS", "IS", "DS", "SE", "IT"]:
    r = lay_ma_mon_tich_luy(ng, hk_max=HK_MAX)
    print(f"\n--- {ng} HK1 -> HK{HK_MAX} ---")
    print(f"bat_buoc = {r['bat_buoc']}")
    print(f"tu_chon  = {r['tu_chon']}")